## EDA

In [ ]:
# Locate the project package from a notebook folder or the project root.
from pathlib import Path
import os
import sys
_pphf_start = Path.cwd().resolve()
_pphf_override = os.environ.get("PPHF_PROJECT_ROOT") or os.environ.get("DP_FHF_PROJECT_ROOT")
_pphf_candidates = ([Path(_pphf_override).expanduser()] if _pphf_override else []) + [_pphf_start, *_pphf_start.parents]
PROJECT_ROOT = next((p.resolve() for p in _pphf_candidates if (p / "PPHF_modules" / "fhf" / "FHF_runner.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Cannot locate PPHF_modules. Set PPHF_PROJECT_ROOT or start Jupyter inside the project.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "Datasets" / 'GEFCOM2017'
import pandas as pd
import numpy as np
from tqdm import tqdm 
import matplotlib.pyplot as plt

In [ ]:
path = str(DATA_DIR) + "/"
gef  = pd.read_csv(f"{path}Gef2017.csv")

# Example data
df = gef.copy()

# Define mapping rules
level_2 = ['CT', 'MASS', 'ME', 'NH', 'RI', 'VT']
level_1 = ['TOTAL']

# Use numpy.select or df.apply; the vectorized form is faster here:
df['level'] = (
    df['zone'].apply(
        lambda x: 2 if x in level_2 else (1 if x in level_1 else 3)
    )
)

df

In [ ]:
df['ts'] = pd.to_datetime(df['ts'].astype(str).str.strip(), format='mixed')
df = df.sort_values(['zone', 'ts']).copy()
df['demand_smooth'] = df.groupby('zone')['demand'].transform(
    lambda values: values.rolling(window=24*7, min_periods=1).mean()
)

# Plot only three levels, selected as needed
levels = sorted(df['level'].unique())#[:3]
zones = sorted(df['zone'].unique())

# Use a fixed color mapping for consistent colors across subplots
cmap = plt.cm.tab20 if len(zones) > 10 else plt.cm.tab10
colors = cmap(np.linspace(0, 1, len(zones)))
zone2color = {z: colors[i % len(colors)] for i, z in enumerate(zones)}

# Three rows and one column
ncols, nrows = 1, 3
fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(17, 9), sharex=True, sharey=False)
if nrows == 1:
    axes = [axes]  # Ensure the axes are iterable

# Collect legend handles
handle_map = {}

for i, level in enumerate(levels):
    ax = axes[i]
    sub = df[df['level'] == level]

    for z in zones:
        zone_data = sub[sub['zone'] == z]
        if not zone_data.empty:
            line, = ax.plot(zone_data['ts'], zone_data['demand_smooth'],
                            color=zone2color[z], linewidth=1.3, label=z, alpha=0.95)
            # Record each zone handle only on its first occurrence
            if z not in handle_map:
                handle_map[z] = line

    ax.set_title(f"Level {level}", fontsize=16, weight='bold')
    #ax.set_xlabel("Time")
    if level == 2:
        ax.set_ylabel("Demand (MW)", fontsize=14, weight='bold')
    ax.tick_params(axis='x', rotation=45)
    ax.grid(alpha=0.3)

    # Black borders around subplots
    for spine in ax.spines.values():
        spine.set_edgecolor('grey')
        spine.set_linewidth(1.2)

# Remove unused subplots if fewer than 3 levels are available
for j in range(len(levels), nrows):
    fig.delaxes(axes[j])

# Overall title
#fig.suptitle("Hourly Electricity Demand by Level (One Week Rolling Mean)", fontsize=16, weight='bold', y=0.98)
#fig.ylabel("Demand (MW)", fontsize=13, weight='bold')
# One legend on the right containing all zones
handles = list(handle_map.values())
labels = list(handle_map.keys())
#fig.legend(handles, labels, loc='center left', bbox_to_anchor=(1, 0.75), frameon=False)#title="Zone"

leg = fig.legend(
    handles, labels,
    #title="Zone",
    loc='center left',
    bbox_to_anchor=(1, 0.8),
    frameon=True,
    fontsize=13#,
    #title_fontsize=11
)

# Add a black border to the legend
frame = leg.get_frame()
frame.set_edgecolor('grey')
frame.set_linewidth(1.2)
frame.set_alpha(1)
frame.set_facecolor('white')

# Leave space for the legend on the right

plt.tight_layout()#rect=[0, 0, 0.85, 0.95]
plt.savefig(str(DATA_DIR / "Hourly_Electricity_Demand.pdf"), format="pdf", bbox_inches="tight")
plt.show()

## Preprocessing

In [ ]:
import pandas as pd
import numpy as np
#import polars as pl
import hierarchicalforecast
from hierarchicalforecast.utils import aggregate
#statsforecast
#MLForecast
#neuralforecast
#
path = str(DATA_DIR) + "/"
#gef  = pd.read_csv(DATA_DIR / str(DATA_DIR / "Gef2017.csv")).iloc[:,:-10]

gef2 = pd.read_csv(DATA_DIR / str(DATA_DIR / "Gef2017.csv"))
gef  = gef2.iloc[:,:-10]
gef['region'] = gef['zone']
gef = gef.loc[gef['zone'].isin(['CT', 'ME', 'NEMASSBOST', 'NH', 'RI', 
                          'SEMASS', 'VT', 'WCMASS']),:]
gef.loc[gef['zone'].str.contains('MASS', na=False), 'zone'] = 'MASS'
gef['total'] = 'Total'
gef.columns = ['ds', 'zone', 'y', 'drybulb', 'dewpnt', 'region', 'total']
SPEC = [
    ["total"],  # Level 1
    ["total", "zone"],  # Level 2
    ["total", "zone", "region"]
]
Y_df, S_df, tags = aggregate(gef, SPEC)
Y_df['levels'] = Y_df['unique_id'].str.count('/') + 1

arr = Y_df['unique_id'].unique()

label2uid = {label: i for i, label in enumerate(arr)}
uid2label = {i: label for label, i in label2uid.items()} 

codes, uniques = pd.factorize(arr)   # codes: integer code for each element
label2id = dict(zip(uniques, range(len(uniques))))

Y_df['partition_id'] = Y_df['unique_id'].map(label2id)

#Y_df = Y_df[Y_df['ds'] >= '2017-04-01']

Y_df

In [ ]:
Y_df.to_csv(str(DATA_DIR / "GEF2017_hierarchy_with_duplicates.csv"), index = False)

In [ ]:
S_df.to_csv(str(DATA_DIR / 'GEF_S_with_duplicates.csv'), index = False)

In [ ]:
import pickle
with open(str(DATA_DIR / "tags_with_duplicates.pkl"), "wb") as f:
    pickle.dump(tags, f)

In [ ]:
def Datacollector(dff, lags:int = 12, ts = range(15)): #48
    from tqdm import tqdm
    arr = {}
    for i in tqdm(ts):
        df_ = dff[dff['partition_id'] == i].copy()
        for j in range(lags):
            df_[f"lags_{(j+1)}"] = df_['y'].shift((j+1)) 
        arr[i] = df_
    return arr
dict_ = Datacollector(Y_df, 48, ts = range(15))
dff = pd.concat([dict_[key] for key in dict_.keys()]).dropna()
dff = dff[dff['ds'] >= '2017-01-01']

#dff_test  =  dff[dff['ds'] >= '2017-01-01']
#dff_train =  dff[dff['ds'] <  '2017-04-01']

dff.to_csv("GEF2017_170101.csv", index = False)

In [ ]:
dff